# Notebook 1

**ClinicalShield v2 · Revision pipeline**

## Purpose
To buld the entire system again.

## What this notebook produces
- `data/corpus/patient_cases.jsonl`
- `data/corpus/rag_corpus.jsonl`
- `data/corpus/manifest.json` — provenance, counts, hashes
- `data/stats/corpus_stats.json` — length distributions, needed by NB02 for length matching




In [15]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────


CONFIG = {
    # "public" = PMC-Patients (no credentialing).  "mimic" = MIMIC-IV-Note.
    "CORPUS_TIER": "public",

    "N_PATIENT_CASES": 1000,
    "N_FDA_LABELS":    3000,



    # MIMIC only — path to the unzipped discharge.csv. Ignored on the public tier.
    "MIMIC_NOTE_PATH": "data/raw/mimic-iv-note/discharge.csv",

    # Reproducibility. Every notebook in this pipeline uses this as its base seed.
    "SEED": 42,

    # openFDA is rate limited without a key. Free key: https://open.fda.gov/apis/authentication/
    "OPENFDA_API_KEY": None,


    "USE_DRIVE":       True,
    "DRIVE_DIR":       "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB":  True,
    "GITHUB_REPO":     "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":   "v2-revision",
    "GIT_USER_NAME":   "Adu-Boahene Bright",
    "GIT_USER_EMAIL":  "baduboahene@st.knust.edu.gh",
}

SEED = CONFIG["SEED"]
print("Corpus tier:", CONFIG["CORPUS_TIER"])
print("Seed:", SEED)

Corpus tier: public
Seed: 42


In [17]:
# ── Cell 2 · Environment ──────────────────────────────────────────────────────
import subprocess, sys, importlib

def ensure(pkg, import_as=None):
    name = import_as or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, mod in [("requests", "requests"), ("pandas", "pandas"),
                 ("numpy", "numpy"), ("tqdm", "tqdm")]:
    ensure(pkg, mod)

import os, json, time, hashlib, random, re
from pathlib import Path
import requests, pandas as pd, numpy as np
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.3 | numpy 2.1.3


## Colab setup



In [18]:
# ── Cell 2b · Drive + GitHub bootstrap ─────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
print("environment:", "Colab" if IN_COLAB else "local")

DRIVE_ROOT = None
REPO_DIR   = None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"])
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive  :", DRIVE_ROOT)

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print("no GITHUB_TOKEN secret found:", type(e).__name__)

    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR  = Path("/content") / repo_name
    if token and not REPO_DIR.exists():
        url = f"https://{token}@github.com/{CONFIG['GITHUB_REPO']}.git"
        r = subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)],
                           capture_output=True, text=True)
        print("clone  :", "ok" if r.returncode == 0 else r.stderr[:200])
    elif REPO_DIR.exists():
        print("clone  : already present")

    if REPO_DIR and REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "config", "user.name",
                        CONFIG["GIT_USER_NAME"]], check=False)
        subprocess.run(["git", "-C", str(REPO_DIR), "config", "user.email",
                        CONFIG["GIT_USER_EMAIL"]], check=False)
        br = CONFIG["GITHUB_BRANCH"]
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", br],
                       check=False, capture_output=True)
        cur = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse",
                              "--abbrev-ref", "HEAD"], capture_output=True, text=True)
        if cur.stdout.strip() != br:
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-qb", br], check=False)
        print("branch :", br)


try:
    import torch
    print("torch  :", torch.__version__,
          "| cuda:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
except ImportError:
    print("torch  : not installed (fine — NB01 needs no GPU)")


if IN_COLAB and CONFIG["OPENFDA_API_KEY"] is None:
    try:
        from google.colab import userdata
        key = userdata.get("OPENFDA_API_KEY")
        if key:
            CONFIG["OPENFDA_API_KEY"] = key
            print("openFDA: key loaded from Colab secrets")
        else:
            print("openFDA: secret empty, using keyless rate limit")
    except Exception:
        print("openFDA: no secret found, using keyless rate limit")
else:
    print("openFDA:", "key already set" if CONFIG["OPENFDA_API_KEY"] else "keyless")

environment: Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive  : /content/drive/MyDrive/ClinicalShield_v2
clone  : already present
branch : v2-revision
torch  : 2.11.0+cu128 | cuda: True | Tesla T4
openFDA: key loaded from Colab secrets


In [19]:
# ── Cell 3 · Directory layout ─────────────────────────────────────────────────
if IN_COLAB and REPO_DIR and REPO_DIR.exists():
    ROOT = REPO_DIR
else:
    ROOT = Path.cwd()
    while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / ".git").exists():
        ROOT = Path.cwd()

DIRS = {
    "raw":    ROOT / "data" / "raw",
    "corpus": ROOT / "data" / "corpus",
    "stats":  ROOT / "data" / "stats",
    "figs":   ROOT / "figures",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
for k, v in DIRS.items():
    print(f"  {k:7s} -> {v.relative_to(ROOT)}")

Project root: /content/ClinicalShield
  raw     -> data/raw
  corpus  -> data/corpus
  stats   -> data/stats
  figs    -> figures


## Patient cases


In [20]:
# ── Cell 4 · Patient case loader ──────────────────────────────────────────────
HF_DATASET = "zhengyun21/PMC-Patients"

def clean(t: str) -> str:
    t = re.sub(r"\s+", " ", str(t))
    return t.strip()

def load_public_cases(n):
    ensure("datasets")
    from datasets import load_dataset
    ds = load_dataset(HF_DATASET, split="train", streaming=True)
    out = []
    for row in ds:
        txt = clean(row.get("patient") or row.get("text") or "")
        if len(txt.split()) < 80:
            continue
        out.append({"text": txt, "source_id": str(row.get("patient_uid", len(out)))})
        if len(out) >= n:
            break
    return out

def load_mimic_cases(n, path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"MIMIC notes not found at {p}. Download MIMIC-IV-Note from PhysioNet "
            "after credentialing, or set CORPUS_TIER='public'."
        )
    df = pd.read_csv(p, usecols=["note_id", "text"], nrows=n * 3)
    out = []
    for _, r in df.iterrows():
        txt = clean(r["text"])
        if len(txt.split()) < 80:
            continue
        out.append({"text": txt, "source_id": str(r["note_id"])})
        if len(out) >= n:
            break
    return out

tier = CONFIG["CORPUS_TIER"]
if tier == "public":
    patient_raw = load_public_cases(CONFIG["N_PATIENT_CASES"])
    provenance  = {"tier": "public", "dataset": HF_DATASET, "redistributable": True}
elif tier == "mimic":
    patient_raw = load_mimic_cases(CONFIG["N_PATIENT_CASES"], CONFIG["MIMIC_NOTE_PATH"])
    provenance  = {"tier": "mimic", "dataset": "MIMIC-IV-Note",
                   "redistributable": False,
                   "note": "DUA prohibits redistribution and third-party API submission"}
else:
    raise ValueError("CORPUS_TIER must be 'public' or 'mimic'")

print(f"Loaded {len(patient_raw)} patient cases from the {tier} tier")
print("Sample length (words):", len(patient_raw[0]["text"].split()))

Loaded 1000 patient cases from the public tier
Sample length (words): 242


## RAG

In [21]:
# ── Cell 5

BASE = "https://api.fda.gov/drug/label.json"
N_DRUGS = 400

FIELDS = ["indications_and_usage","dosage_and_administration","contraindications",
          "warnings_and_cautions","drug_interactions","boxed_warning",
          "adverse_reactions","use_in_specific_populations"]


def fetch_top_generics(n, api_key=None):
    """Rank generic names by how many label documents openFDA holds for them."""
    params = {"count": "openfda.generic_name.exact", "limit": 1000}
    if api_key:
        params["api_key"] = api_key
    r = requests.get(BASE, params=params, timeout=60)
    r.raise_for_status()
    terms = r.json().get("results", [])

    out = []
    for t in terms:
        name = t["term"].strip().lower()

        if any(sep in name for sep in [",", " and ", "/", ";", "+"]):
            continue
        if len(name) < 4 or len(name) > 40:
            continue
        if not name.replace(" ", "").replace("-", "").isalpha():
            continue
        out.append({"name": name, "label_count": int(t["count"])})
        if len(out) >= n:
            break
    return out


def fetch_label(drug, api_key=None, retries=3):
    params = {"search": f'openfda.generic_name:"{drug}"', "limit": 1}
    if api_key:
        params["api_key"] = api_key
    for attempt in range(retries):
        try:
            r = requests.get(BASE, params=params, timeout=30)
            if r.status_code == 404:
                return None
            if r.status_code == 429:
                time.sleep(2 ** attempt + 1)
                continue
            r.raise_for_status()
            res = r.json().get("results", [])
            return res[0] if res else None
        except requests.RequestException:
            if attempt == retries - 1:
                return None
            time.sleep(2 ** attempt)
    return None


# ── build the sampling frame ──────────────────────────────────────────────────
API_KEY = CONFIG["OPENFDA_API_KEY"]
print("openFDA key:", "present" if API_KEY else "absent (keyless limits)")

DRUG_FRAME = fetch_top_generics(N_DRUGS, API_KEY)
DRUGS = [d["name"] for d in DRUG_FRAME]

print(f"sampling frame: {len(DRUGS)} generics derived from openFDA")
print(f"  most frequent : {DRUGS[0]} ({DRUG_FRAME[0]['label_count']} labels)")
print(f"  least frequent: {DRUGS[-1]} ({DRUG_FRAME[-1]['label_count']} labels)")
print(f"  examples      : {', '.join(DRUGS[:8])}")

# ── retrieve label sections ───────────────────────────────────────────────────
DELAY = 0.05 if API_KEY else 0.2
print(f"\nquerying {len(DRUGS)} drugs x {len(FIELDS)} sections "
      f"(~{len(DRUGS)*DELAY/60:.1f} min minimum)")

rag_raw, failed = [], []
seen = set()

for drug in tqdm(DRUGS, desc="openFDA"):
    label = fetch_label(drug, API_KEY)
    if label is None:
        failed.append(drug)
        continue
    for field in FIELDS:
        val = label.get(field)
        if not val:
            continue
        txt = clean(" ".join(val) if isinstance(val, list) else val)
        if len(txt.split()) < 40:
            continue
        key = hashlib.md5(txt[:300].encode()).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        rag_raw.append({"text": txt, "drug": drug, "section": field,
                        "doc_type": "drug_label"})
    time.sleep(DELAY)
    if len(rag_raw) >= CONFIG["N_FDA_LABELS"]:
        print(f"\nhit N_FDA_LABELS cap at '{drug}' "
              f"({DRUGS.index(drug)+1}/{len(DRUGS)} drugs processed)")
        break

n_drugs = len({d["drug"] for d in rag_raw})
print(f"\nRetrieved {len(rag_raw)} label sections across {n_drugs} drugs "
      f"({len(rag_raw)/max(n_drugs,1):.1f} sections/drug)")
print(f"Section coverage: {len({d['section'] for d in rag_raw})}/{len(FIELDS)}")
if failed:
    print(f"No label for {len(failed)}: {', '.join(failed[:15])}"
          + (" ..." if len(failed) > 15 else ""))

openFDA key: present
sampling frame: 400 generics derived from openFDA
  most frequent : zinc oxide (1956 labels)
  least frequent: felodipine (29 labels)
  examples      : zinc oxide, alcohol, acetaminophen, salicylic acid, ibuprofen, menthol, benzalkonium chloride, sodium fluoride

querying 400 drugs x 8 sections (~0.3 min minimum)


openFDA:   0%|          | 0/400 [00:00<?, ?it/s]


Retrieved 1604 label sections across 336 drugs (4.8 sections/drug)
Section coverage: 8/8


In [22]:
# ── Cell 6 ────────────────────────────────
def doc_id(prefix, i):   return f"{prefix}_{i:06d}"
def group_of(prefix, i): return f"g_{prefix}_{i:06d}"

patient_cases = []
for i, rec in enumerate(patient_raw):
    wc = len(rec["text"].split())
    patient_cases.append({
        "doc_id":   doc_id("case", i),
        "group_id": group_of("case", i),
        "role":     "patient_case",
        "text":     rec["text"],
        "source_id": rec["source_id"],
        "word_count": wc,
        "char_count": len(rec["text"]),
        "is_derivative": False,
        "parent_doc_id": None,
    })

rag_corpus = []
for i, rec in enumerate(rag_raw):
    wc = len(rec["text"].split())
    rag_corpus.append({
        "doc_id":   doc_id("rag", i),
        "group_id": group_of("rag", i),
        "role":     "rag_document",
        "text":     rec["text"],
        "drug":     rec["drug"],
        "section":  rec["section"],
        "doc_type": rec["doc_type"],
        "word_count": wc,
        "char_count": len(rec["text"]),
        "label":    0,
        "is_derivative": False,
        "parent_doc_id": None,
    })

assert len({d["group_id"] for d in rag_corpus}) == len(rag_corpus), "group_id collision"
assert len({d["group_id"] for d in patient_cases}) == len(patient_cases), "group_id collision"

print(f"patient cases : {len(patient_cases):>5}  unique groups: {len({d['group_id'] for d in patient_cases}):>5}")
print(f"rag documents : {len(rag_corpus):>5}  unique groups: {len({d['group_id'] for d in rag_corpus}):>5}")

patient cases :  1000  unique groups:  1000
rag documents :  1604  unique groups:  1604


In [23]:
# ── Cell 7 · Length distributions ─────────────────────────────────────────────
def describe(docs, name):
    w = np.array([d["word_count"] for d in docs])
    st = {
        "name": name, "n": int(len(w)),
        "mean": float(w.mean()), "std": float(w.std()),
        "min": int(w.min()), "max": int(w.max()),
        "p05": float(np.percentile(w, 5)),  "p25": float(np.percentile(w, 25)),
        "median": float(np.median(w)),
        "p75": float(np.percentile(w, 75)), "p95": float(np.percentile(w, 95)),
    }
    print(f"\n{name}  (n={st['n']})")
    print(f"  median {st['median']:>8.0f}   mean {st['mean']:>8.0f}   sd {st['std']:>8.0f}")
    print(f"  p05 {st['p05']:>6.0f} | p25 {st['p25']:>6.0f} | "
          f"p75 {st['p75']:>6.0f} | p95 {st['p95']:>6.0f}")
    return st

stats = {
    "patient_cases": describe(patient_cases, "Patient cases"),
    "rag_corpus":    describe(rag_corpus,    "RAG corpus"),
}

stats["length_matching_target"] = {
    "note": "NB02 must draw injected-document lengths from this distribution",
    "source": "rag_corpus",
    "percentiles": {k: stats["rag_corpus"][k]
                    for k in ["p05","p25","median","p75","p95"]},
}


Patient cases  (n=1000)
  median      364   mean      436   sd      285
  p05    133 | p25    255 | p75    549 | p95    946

RAG corpus  (n=1604)
  median      367   mean      652   sd      791
  p05     53 | p25    135 | p75    871 | p95   2159


In [24]:
# ── Cell 8 ─────────────────────────────────────
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path

def sha256(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

p_cases = write_jsonl(DIRS["corpus"] / "patient_cases.jsonl", patient_cases)
p_rag   = write_jsonl(DIRS["corpus"] / "rag_corpus.jsonl",    rag_corpus)

manifest = {
    "notebook": "01_corpus_construction",
    "seed": SEED,
    "config": {k: v for k, v in CONFIG.items() if k != "OPENFDA_API_KEY"},
    "provenance": {
        "patient_cases": provenance,
        "rag_corpus": {"source": "openFDA drug label API",
                       "endpoint": BASE,
                       "drugs_requested": len(set(DRUGS)),
                       "drugs_retrieved": len({d["drug"] for d in rag_corpus}),
                       "drugs_failed": failed,
                       "sections": FIELDS,
                       "redistributable": True},
    },
    "counts": {
        "patient_cases": len(patient_cases),
        "rag_documents": len(rag_corpus),
        "patient_groups": len({d["group_id"] for d in patient_cases}),
        "rag_groups": len({d["group_id"] for d in rag_corpus}),
    },
    "files": {
        "patient_cases.jsonl": {"rows": len(patient_cases), "sha256_16": sha256(p_cases)},
        "rag_corpus.jsonl":    {"rows": len(rag_corpus),    "sha256_16": sha256(p_rag)},
    },
}


manifest["provenance"]["rag_corpus"]["drug_sampling_frame"] = {
    "method": "top-N generic names by openFDA label document count",
    "endpoint_params": {"count": "openfda.generic_name.exact", "limit": 1000},
    "filters": "single-ingredient only; 4-40 chars; alphabetic",
    "n_requested": N_DRUGS,
    "n_used": len(DRUGS),
    "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "drugs": DRUG_FRAME,
}

with open(DIRS["corpus"] / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
with open(DIRS["stats"] / "corpus_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("written:")
for p in [p_cases, p_rag,
          DIRS["corpus"] / "manifest.json",
          DIRS["stats"] / "corpus_stats.json"]:
    print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size/1024:.0f} KB)")

print(f"\ndrug sampling frame frozen: {len(DRUG_FRAME)} generics")

written:
  data/corpus/patient_cases.jsonl  (3012 KB)
  data/corpus/rag_corpus.jsonl  (7478 KB)
  data/corpus/manifest.json  (39 KB)
  data/stats/corpus_stats.json  (1 KB)

drug sampling frame frozen: 400 generics


In [26]:
# ── Cell 8b ────────────────────
import shutil

if IN_COLAB and CONFIG["USE_DRIVE"] and DRIVE_ROOT:
    for sub in ["corpus", "stats"]:
        dst = DRIVE_ROOT / "data" / sub
        dst.mkdir(parents=True, exist_ok=True)
        for f in (ROOT / "data" / sub).glob("*"):
            shutil.copy2(f, dst / f.name)
    print("mirrored to Drive:", DRIVE_ROOT / "data")
else:
    print("Drive mirror skipped")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    gi = ROOT / ".gitignore"
    rules = ["data/raw/", "data/corpus/*.jsonl", "*.pt", "*.ckpt", "__pycache__/"]


    existing = [line.strip() for line in gi.read_text().splitlines()] if gi.exists() else []


    missing = [r for r in rules if r not in existing]

    if missing:

        with open(gi, "a") as f:

            if gi.exists() and gi.stat().st_size > 0:
                f.write("\n")
            f.write("\n".join(missing) + "\n")

        print("updated .gitignore:", missing)

    paths = ["data/corpus/manifest.json", "data/stats/corpus_stats.json", ".gitignore"]
    subprocess.run(["git", "-C", str(ROOT), "add", "-f", *paths], check=False)
    st = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        msg = (f"NB01: corpus built ({len(rag_corpus)} rag docs, "
               f"{len(patient_cases)} cases, tier={CONFIG['CORPUS_TIER']})")
        subprocess.run(["git", "-C", str(ROOT), "commit", "-q", "-m", msg], check=False)
        pr = subprocess.run(["git", "-C", str(ROOT), "push", "-q", "origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push   :", "ok" if pr.returncode == 0 else pr.stderr[:300])
    else:
        print("push   : nothing to commit")
else:
    print("GitHub push skipped")

mirrored to Drive: /content/drive/MyDrive/ClinicalShield_v2/data
push   : nothing to commit


In [27]:
# ── Cell 9 ─────────────────────────────────
print("=" * 62)
print("NB01 — CORPUS CONSTRUCTION")
print("=" * 62)
print(f"corpus tier            : {CONFIG['CORPUS_TIER']}")
print(f"seed                   : {SEED}")
print()
print(f"patient cases          : {len(patient_cases)}")
print(f"rag documents          : {len(rag_corpus)}")
print(f"unique drugs           : {len({d['drug'] for d in rag_corpus})}")
print(f"label sections covered : {len({d['section'] for d in rag_corpus})}")
print()
print(f"patient group_ids      : {len({d['group_id'] for d in patient_cases})}")
print(f"rag group_ids          : {len({d['group_id'] for d in rag_corpus})}")
print(f"group_id collisions    : 0 (asserted)")
print()
r = stats["rag_corpus"]; pc = stats["patient_cases"]
print("length (words)         median      mean        sd")
print(f"  patient cases      {pc['median']:>9.0f} {pc['mean']:>9.0f} {pc['std']:>9.0f}")
print(f"  rag documents      {r['median']:>9.0f} {r['mean']:>9.0f} {r['std']:>9.0f}")
print()
print("NB02 length-match target (rag corpus percentiles):")
for k, v in stats["length_matching_target"]["percentiles"].items():
    print(f"  {k:>6} : {v:>8.0f} words")
print()
print(f"sha256(rag_corpus.jsonl)[:16] = {manifest['files']['rag_corpus.jsonl']['sha256_16']}")
print("=" * 62)
print("NB01 COMPLETE — ready for NB02 (attack generation)")
print("=" * 62)

NB01 — CORPUS CONSTRUCTION
corpus tier            : public
seed                   : 42

patient cases          : 1000
rag documents          : 1604
unique drugs           : 336
label sections covered : 8

patient group_ids      : 1000
rag group_ids          : 1604
group_id collisions    : 0 (asserted)

length (words)         median      mean        sd
  patient cases            364       436       285
  rag documents            367       652       791

NB02 length-match target (rag corpus percentiles):
     p05 :       53 words
     p25 :      135 words
  median :      367 words
     p75 :      871 words
     p95 :     2159 words

sha256(rag_corpus.jsonl)[:16] = cbd62de16bd85649
NB01 COMPLETE — ready for NB02 (attack generation)
